In [1]:
import pandas as pd
import numpy as np

# 1. Load the data
sensors = pd.read_csv('../data/raw/secom.data', sep=' ', header=None)
labels = pd.read_csv('../data/raw/secom_labels.data', sep=' ', header=None)
labels.columns = ['Result', 'Timestamp']

# 2. Combine and clean labels
df = pd.concat([labels, sensors], axis=1)
df['Result'] = df['Result'].replace(-1, 0)

# 3. Identify dead weight columns
missing_pct = df.isnull().mean()
high_missing_cols = missing_pct[missing_pct > 0.50]

unique_counts = df.nunique()
constant_cols = unique_counts[unique_counts == 1]

# 4. Drop the dead weight
cols_to_drop = list(set(high_missing_cols.index).union(set(constant_cols.index)))
df_clean = df.drop(columns=cols_to_drop)

print(f"Cleaned Initial Shape: {df_clean.shape}")

Cleaned Initial Shape: (1567, 448)


In [2]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score

# 1. The 3-Way Split
X = df_clean.drop(columns=['Timestamp', 'Result'])
y = df_clean['Result']
X.columns = X.columns.astype(str)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42)

# 2. The Zero-Leakage Pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('imputer', KNNImputer(n_neighbors=5)),
    ('selector', SelectFromModel(xgb.XGBClassifier(random_state=42, max_depth=3))),
    ('model', xgb.XGBClassifier(random_state=42, eval_metric='aucpr', tree_method='hist', n_estimators=200, subsample=0.8, colsample_bytree=0.8))
])

# 3. The Refined Grid Search (with the syntax fix applied)
param_grid = {
    'selector__max_features': [20, 50, 100],
    'selector__threshold': [-np.inf], # No quotes around -np.inf
    'model__scale_pos_weight': [10, 14, 18],
    'model__max_depth': [3, 4]
}

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring='average_precision', cv=3, verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

champion_pipe = grid_search.best_estimator_

# 4. Evaluate Threshold on Validation
val_probs = champion_pipe.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_threshold = thresholds[np.argmax(f1_scores)]

# 5. Final Honest Test Score
test_probs = champion_pipe.predict_proba(X_test)[:, 1]
final_test_predictions = (test_probs >= optimal_threshold).astype(int)

print("=== TRUE PRODUCTION PERFORMANCE (TEST SET) ===")
print(classification_report(y_test, final_test_predictions))

Fitting 3 folds for each of 18 candidates, totalling 54 fits
=== TRUE PRODUCTION PERFORMANCE (TEST SET) ===
              precision    recall  f1-score   support

           0       0.95      0.91      0.93       220
           1       0.24      0.38      0.29        16

    accuracy                           0.88       236
   macro avg       0.60      0.64      0.61       236
weighted avg       0.90      0.88      0.89       236



In [3]:
import joblib

# Extract the names of the sensors the model decided to keep
winning_selector = champion_pipe.named_steps['selector']
selected_sensor_names = X_train.columns[winning_selector.get_support()]

# Package the asset
secom_production_asset = {
    'pipeline': champion_pipe,
    'optimal_threshold': optimal_threshold,
    'sensor_names': selected_sensor_names.tolist() 
}

# Export it
joblib.dump(secom_production_asset, 'secom_xgboost_production_v1.pkl')
print("SUCCESS: Model successfully compiled and saved to secom_xgboost_production_v1.pkl")

SUCCESS: Model successfully compiled and saved to secom_xgboost_production_v1.pkl
